## **變異數分析 (ANOVA) 與假設檢定**：跨機台差異判斷

- 目標：掌握假設檢定與單因子變異數分析（One-way ANOVA）的理論與程式碼實作。學會利用 scipy.stats 驗證資料是否符合常態性與變異數同質性前提，並使用 statsmodels 進行 ANOVA 檢定與事後檢定（Tukey's HSD），**量化找出到底是哪一台測試機台發生異常**。


### 1. 統計假設檢定架構與**前提驗證**

- 實作：
    - 產線上有三台不同的高頻測試機台（Tool_A, Tool_B, Tool_C）同時在跑同一款新晶圓的頻寬測試。我們抽取了每台機台的測試數據，在進行變異數分析前，必須先驗證兩個關鍵前提：
        - 1. **常態性檢定（Shapiro-Wilk Test）**
        - 2. **變異數同質性檢定（Levene's Test）**


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# 模擬三台測試機台的頻寬測試數據（GHz）
# 故意讓 Tool_C 的平均值顯著偏低，模擬探針老化磨損
np.random.seed(42)
tool_A = np.random.normal(loc=28.1, scale=0.4, size=30)
tool_B = np.random.normal(loc=28.0, scale=0.4, size=30)
tool_C = np.random.normal(loc=27.4, scale=0.5, size=30)  # 異常機台

# 整合為 Pandas DataFrame
data_dict = {
    "bandwidth": np.concatenate([tool_A, tool_B, tool_C]),
    "tool": ["Tool_A"] * 30 + ["Tool_B"] * 30 + ["Tool_C"] * 30,
}
df_tools = pd.DataFrame(data_dict)

# 前提驗證 A：常態性檢定 (Shapiro-Wilk Test)
# H0: 資料符合常態分佈；若 p-value > 0.05 則通過常態性假設
print(">>> 執行常態性檢定 (Shapiro-Wilk Test)：")
for name, group in df_tools.groupby("tool"):
    stat, p_val = stats.shapiro(group["bandwidth"])
    print(
        f"{name} - 統計量: {stat:.4f}, p-value: {p_val:.4f} -> {'通過常態性' if p_val > 0.05 else '未通過'}"
    )

# 前提驗證 B：變異數同質性檢定 (Levene's Test)
# H0: 各組變異數相等；若 p-value > 0.05 則通過同質性假設
stat_levene, p_levene = stats.levene(tool_A, tool_B, tool_C)
print(
    f"\n>>> 執行變異數同質性檢定 (Levene Test)：\n, p-value: {p_levene:.4f} -> {'通過變異數同質性' if p_levene > 0.05 else '未通過'}"
)

## 2. 使用 statsmodels 執行單因子變異數分析 (One-way ANOVA)
- 核心：
    - **虛無假設 (\(H_{0}\))**：\(\mu_A = \mu_B = \mu_C\)（**所有機台的量測均值完全相同，無顯著差異**）。
    - **對立假設 (\(H_{1}\))**：**至少有兩台機台之間的量測均值存在顯著差異**。
    - 若 F-檢定的 \(p\text{-value} < 0.05\)，則拒絕 \(H_{0}\)，代表**機台間有顯著差異**，需啟動工程調查。

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

# 建立普通最小平方法 (OLS) 模型
# 語法 'bandwidth ~ tool' 代表 bandwidth 是應變數，tool 是自變數
model = ols("bandwidth ~ tool", data=df_tools).fit()

# 進行 ANOVA 分析並印出摘要表
anova_table = sm.stats.anova_lm(model, typ=2)
print("=" * 60)
print("變異數分析 (ANOVA) 摘要表：")
print(anova_table)
print("=" * 60)

p_anova = anova_table["PR(>F)"].iloc[0]
if p_anova < 0.05:
    print(f"結論：p-value ({p_anova:.4e}) < 0.05，強烈拒絕虛無假設！")
    print("統計學證據顯示：不同機台間存在【顯著良率/頻寬差異】，必須進行事後檢定。")
else:
    print("結論：p-value > 0.05，無法拒絕虛無假設。機台間表現一致。")

## 3. **事後檢定（Tukey's HSD）與盒鬚圖視覺化**

- 實作：ANOVA 只能告訴我們「有差異」，但無法指出是「誰跟誰有差異」。我們必須透過 **Tukey's HSD 進行兩兩成對比較（Pairwise Comparisons）**，精準鎖定異常源頭。


In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# 執行 Tukey's HSD 事後檢定
tukey = pairwise_tukeyhsd(
    endog=df_tools["bandwidth"], groups=df_tools["tool"], alpha=0.05
)
print("\n Tukey's HSD 事後兩兩比較結果：")
print(tukey)

# 使用 Seaborn 繪製盒鬚圖 (Boxplot) 直觀呈現差異
plt.figure(figsize=(7, 4.5))
sns.boxplot(x="tool", y="bandwidth", data=df_tools, palette="Set2", width=0.5)
sns.stripplot(
    x="tool", y="bandwidth", data=df_tools, color="black", alpha=0.3, jitter=0.1
)

plt.title("跨測試機台（Tools）頻寬數據分佈與假設檢定對照", fontsize=12)
plt.xlabel("測試機台型號 (Tool ID)")
plt.ylabel("量測頻寬 (GHz)")
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

- 總結：在 NPI 新產品引進階段，當跨產線、跨機台的良率或電性數據出現波動時，工程師不能單憑經驗瞎猜。在我的學習與專案實作中，我會撰寫 Python 指令稿進行 單因子變異數分析 (One-way ANOVA)。在分析前，我會嚴謹地使用 scipy.stats 執行 Shapiro-Wilk 常態性檢定 與 Levene 變異數同質性檢定，確保資料符合統計假設。當 statsmodels 的 OLS 模型計算出 \(p\text{-value}\) 顯著拒絕虛無假設後，我會進一步啟動 Tukey's HSD 事後檢定 進行兩兩對比。這能幫我用客觀的統計學證據，精準揪出到底是哪一台機台（如本例中的 Tool_C）的數據產生偏離，從而引導硬體團隊針對該機台的探針或針床進行更換，大幅提升製程除錯的效率。
